In [1]:
import sys
import os
from unittest.mock import MagicMock

# 1. Mock boto3 BEFORE importing session to avoid DynamoDB connection attempts
sys.modules["boto3"] = MagicMock()

# 2. Setup path to import from parent directory
sys.path.append(os.path.abspath('..'))

import session as session_module
from session import ChatSession
from messages import ChatMessage

import nest_asyncio
nest_asyncio.apply()

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 2


{"level":"INFO","location":"set_max_size:61","message":"Max size for namespace pdf_utils set to 100","timestamp":"2026-01-01 17:26:45,570+0000","service":"service_undefined","taskName":"Task-31"}
{"level":"INFO","location":"set_max_size:61","message":"Max size for namespace web_reader set to 150","timestamp":"2026-01-01 17:26:45,800+0000","service":"service_undefined","taskName":"Task-31"}
{"level":"INFO","location":"set_max_size:61","message":"Max size for namespace ytsubs set to 200","timestamp":"2026-01-01 17:26:45,805+0000","service":"service_undefined","taskName":"Task-31"}


In [2]:
# 3. Create a mock Slack client
mock_client = MagicMock()

# Mock users_info (called in ChatSession.__init__)
mock_client.users_info.return_value = {
    "user": {
        "real_name": "Test User"
    }
}

# Mock conversations_replies (for conversation history)
mock_client.conversations_replies.return_value = {
    "messages": [
        {"user": "user1", "text": "Hello Bot", "ts": "1000.0"}
    ]
}

# Mock chat_postMessage and chat_update
mock_client.chat_postMessage.return_value = {"ts": "1234.5678"}
mock_client.chat_update.return_value = {"ok": True}

# 4. Define a simple Logger class for the notebook
class NotebookLogger:
    def info(self, msg): print(f"[INFO] {msg}")
    def error(self, msg): print(f"[ERROR] {msg}")
    def debug(self, msg): print(f"[DEBUG] {msg}")
    def warning(self, msg): print(f"[WARN] {msg}")

In [3]:
# 5. Mock the LLM completion to avoid API calls
def mock_stream_response(*args, **kwargs):
    # Yield chunks to simulate streaming response
    chunk1 = MagicMock()
    chunk1.choices = [MagicMock(delta=MagicMock(content="Hello! ", reasoning_content="Thinking...\n"))]
    yield chunk1

    chunk2 = MagicMock()
    chunk2.choices = [MagicMock(delta=MagicMock(content="I am a mocked assistant.", reasoning_content=None))]
    yield chunk2

session_module.completion = MagicMock(side_effect=mock_stream_response)

# 6. Initialize the ChatSession
session = ChatSession(
    user_id='user1',
    channel_id='channel1',
    thread_ts='thread1',
    client=mock_client,
    logger=NotebookLogger()
)

print(f"Session initialized for user: {session.user_name}")

# 7. Run a test message processing
print("\n--- Processing message ---")
session.agent='search'
session._generate_from_adk_agent([
            ChatMessage.from_user(
                "Hi bot"
            )
        ])

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Session initialized for user: Test User

--- Processing message ---
[DEBUG] Generating response using ADK agent: search
{"level":"INFO","location":"run_agent_streaming:639","message":{"current_message":"Hello! 👋 I'm **SushiBot** 🍣. I'm your friend here on Slack, ready to help you with whatever you need!\n\nHow can I help you today? Feel free to ask me anything! 😊","total_posted":"","message_ts":"1234.5678","last_update_time":1767288409.926172,"error":null},"timestamp":"2026-01-01 17:26:49,926+0000","service":"service_undefined","taskName":"Task-7"}
{"level":"INFO","location":"_generate_from_adk_agent:649","message":{"current_message":"Hello! 👋 I'm **SushiBot** 🍣. I'm your friend here on Slack, ready to help you with whatever you need!\n\nHow can I help you today? Feel free to ask me anything! 😊","total_posted":"","message_ts":"1234.5678","last_update_time":1767288409.926172,"error":null},"timestamp":"2026-01-01 17:26:49,928+0000","service":"service_undefined","taskName":"Task-6"}
{"lev